# RLHF 학습 구현 — HuggingFace TRL
**참고 문서**: <https://huggingface.co/docs/trl/example_overview>

# 1. 필요 라이브러리 준비

In [1]:
!pip install -q -U "trl>=1.7" transformers datasets accelerate

In [2]:
import os, gc, torch
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"   # PPO experimental import 경고 숨김

import trl, transformers
print("trl:", trl.__version__, "| transformers:", transformers.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"GPU 여유 메모리: {free/1e9:.2f} / {total/1e9:.2f} GB")


# 단계 사이에 메모리를 비우는 함수 (OOM 방지)
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

trl: 1.7.1 | transformers: 5.13.0
device: cuda
GPU 여유 메모리: 6.26 / 15.64 GB


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2" # 가볍게 실습할 수 있는 `distilgpt2` 를 사용
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [4]:
# 프롬프트 템플릿 정의 (시나리오: 고객 문의 → 상담원 답변)
PROMPT_TEMPLATE = "### Customer: {q}\n### Agent:"

# 답변 생성 함수 정의
def generate(model, question, max_new_tokens=24):
    prompt = PROMPT_TEMPLATE.format(q=question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True, top_p=0.9, temperature=0.8,
        pad_token_id=tokenizer.pad_token_id,
    )
    answer = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return answer.strip()

In [5]:
print(generate(base_model, "Can I get a refund?"))

Thank you for your service! Thank you!
*Note: This information is for the purposes of this article, and


학습 전 베이스 모델이 엉뚱하게 이어쓰기만 하는 모습. 이걸 SFT로 고쳐줄 겁니다.

In [6]:
del base_model
clear_memory()

# 2. 데이터셋 확인

RLHF는 단계마다 다른 데이터가 필요합니다.

- **SFT용**: 질문 → 좋은 답변
- **Reward Model용**: 같은 질문에 대한 (chosen=좋은 답변) vs (rejected=나쁜 답변) 선호쌍
- **PPO용**: 질문(프롬프트)만 있으면 됨 (답변은 학습 중 모델이 직접 생성)

In [7]:
# (질문, 좋은 답변) — SFT 용
QA = [
    ("How do I reset my password?", "Sure! Go to Settings, tap Reset Password, and follow the email link."),
    ("What are your business hours?", "We're happy to help from 9am to 6pm, Monday through Friday."),
    ("Can I get a refund?", "Of course, I'd be glad to help you start a refund right away."),
    ("The app keeps crashing.", "I'm sorry about that! Please update to the latest version."),
    ("Where is my order?", "Let me check that for you — your order arrives tomorrow."),
    ("How do I contact support?", "You can reach our friendly team anytime at support@example.com."),
]

# (질문, chosen=친절, rejected=불친절) — Reward Model 용
PREF = [
    ("Can I get a refund?", "Of course, I'd be glad to help you start a refund right away.",
     "No. Refunds are not my problem, figure it out yourself."),
    ("The app keeps crashing.", "I'm so sorry! Let's fix this together — please try reinstalling.",
     "Works fine for me. Not my fault your phone is broken."),
    ("How do I reset my password?", "Happy to help! Just tap Reset Password on the login screen.",
     "Read the manual. I don't have time for this."),
    ("Where is my order?", "Great question! Let me look that up and update you right now.",
     "How should I know where your stuff is? Stop bothering me."),
    ("What are your business hours?", "We're open 9am to 6pm on weekdays and always glad to help!",
     "Whenever we feel like it. Go away."),
    ("How do I contact support?", "You can always reach our friendly team at support@example.com!",
     "Don't. We don't want to hear from you."),
]

print("SFT 예시:", len(QA), "개 | 선호쌍:", len(PREF), "개")

SFT 예시: 6 개 | 선호쌍: 6 개


SFT 데이터 한 개를 확인해봅니다.

In [8]:
q, a = QA[2]
print("질문 :", q)
print("답변 :", a)

질문 : Can I get a refund?
답변 : Of course, I'd be glad to help you start a refund right away.


Reward Model 데이터(선호쌍) 한 개를 확인해봅니다.
같은 질문에 대해 **chosen(친절)** 이 **rejected(불친절)** 보다 좋다고 알려주는 데이터입니다.

In [9]:
q, chosen, rejected = PREF[0]
print("질문     :", q)
print("chosen   :", chosen)
print("rejected :", rejected)

질문     : Can I get a refund?
chosen   : Of course, I'd be glad to help you start a refund right away.
rejected : No. Refunds are not my problem, figure it out yourself.


# 3. SFT(Supervised Fine-Tuning)

사람이 쓴 좋은 답변(`QA`)을 그대로 흉내내도록 `distilgpt2` 를 미세조정합니다.

In [10]:
# 학습 데이터셋 구성
from datasets import Dataset

sft_dataset = Dataset.from_list([
    {"text": PROMPT_TEMPLATE.format(q=q) + " " + a + tokenizer.eos_token}
    for q, a in QA
])
print(sft_dataset[0]["text"])

### Customer: How do I reset my password?
### Agent: Sure! Go to Settings, tap Reset Password, and follow the email link.<|endoftext|>


TRL의 `SFTTrainer` 로 학습합니다.

In [29]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="sft_distilgpt2",
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=5,
    report_to="none",
    bf16=False, fp16=False,
)

sft_trainer = SFTTrainer(
    model=MODEL_NAME,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
)
sft_trainer.train()
sft_trainer.save_model("sft_distilgpt2")
tokenizer.save_pretrained("sft_distilgpt2")
print("SFT 완료 → ./sft_distilgpt2 저장")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss
5,3.487170
10,2.349467
15,1.595329
20,1.211183
25,0.941383
30,0.752323
35,0.649066
40,0.609637


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SFT 완료 → ./sft_distilgpt2 저장


학습 후 생성 결과를 봅니다. 친절한 답변을 따라 합니다.

In [31]:
sft_trainer.model.eval()
for q in ["Can I get a refund?", "How do I contact support?"]:
    print(f"Q: {q}\nA: {generate(sft_trainer.model, q)}\n")

Q: Can I get a refund?
A: Sure! Go to Settings, tap on My Account, and follow the email link.

Q: How do I contact support?
A: We're happy to help from 9am to 6pm.



In [26]:
del sft_trainer
clear_memory()

# 4. Reward Model

`(chosen ≻ rejected)` 선호쌍으로 **보상모델(RM)** 을 학습합니다.

In [14]:
# Reward Model 불러오기
from transformers import AutoModelForSequenceClassification

reward_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
reward_model.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: distilgpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


선호쌍 데이터를 `prompt / chosen / rejected` 형식으로 만듭니다.

In [15]:
reward_dataset = Dataset.from_list([
    {"prompt": PROMPT_TEMPLATE.format(q=q), "chosen": " " + chosen, "rejected": " " + rejected}
    for q, chosen, rejected in PREF
])
print(reward_dataset[0])

{'prompt': '### Customer: Can I get a refund?\n### Agent:', 'chosen': " Of course, I'd be glad to help you start a refund right away.", 'rejected': ' No. Refunds are not my problem, figure it out yourself.'}


TRL의 `RewardTrainer` 로 학습합니다.

In [16]:
from trl import RewardConfig, RewardTrainer

reward_config = RewardConfig(
    output_dir="reward_distilgpt2",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=1e-4,
    logging_steps=5,
    report_to="none",
    bf16=False, fp16=False,
    gradient_checkpointing=False,
)

reward_trainer = RewardTrainer(
    model=reward_model,
    args=reward_config,
    train_dataset=reward_dataset,
    processing_class=tokenizer,
)
reward_trainer.train()
reward_trainer.save_model("reward_distilgpt2")
tokenizer.save_pretrained("reward_distilgpt2")
print("Reward Model 완료 → ./reward_distilgpt2 저장")

Adding EOS to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Filtering train >1024 tokens:   0%|          | 0/6 [00:00<?, ? examples/s]

Step,Training Loss
5,0.252713
10,0.000432
15,0.000030
20,0.000012


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Reward Model 완료 → ./reward_distilgpt2 저장


보상모델이 잘 배웠는지 확인합니다. **친절한 답변이 불친절한 답변보다 높은 점수**를 받아야 합니다.

In [17]:
def reward_score(question, answer, model):
    text = PROMPT_TEMPLATE.format(q=question) + " " + answer
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        return model(**inputs).logits[0].item()

reward_trainer.model.eval()
for q, chosen, rejected in PREF[:3]:
    print(f"Q: {q}")
    print(f"  chosen  ({reward_score(q, chosen,  reward_trainer.model):+.2f}): {chosen}")
    print(f"  rejected({reward_score(q, rejected, reward_trainer.model):+.2f}): {rejected}\n")

Q: Can I get a refund?
  chosen  (+7.32): Of course, I'd be glad to help you start a refund right away.
  rejected(-4.29): No. Refunds are not my problem, figure it out yourself.

Q: The app keeps crashing.
  chosen  (+8.65): I'm so sorry! Let's fix this together — please try reinstalling.
  rejected(-2.46): Works fine for me. Not my fault your phone is broken.

Q: How do I reset my password?
  chosen  (+7.33): Happy to help! Just tap Reset Password on the login screen.
  rejected(-6.46): Read the manual. I don't have time for this.



In [18]:
del reward_trainer, reward_model
clear_memory()

# 5. PPO(Proximal Policy Optimization)

마지막으로, SFT 정책이 **보상모델 점수를 최대화**하도록 PPO로 강화학습합니다.

In [19]:
# PPO 학습을 위한 4개 모델 불러오기
from trl.experimental.ppo import PPOTrainer, PPOConfig

policy       = AutoModelForCausalLM.from_pretrained("sft_distilgpt2")            # 학습 대상
ref_policy   = AutoModelForCausalLM.from_pretrained("sft_distilgpt2")            # 고정 기준
reward_model = AutoModelForSequenceClassification.from_pretrained("reward_distilgpt2", num_labels=1)
value_model  = AutoModelForSequenceClassification.from_pretrained("sft_distilgpt2", num_labels=1)

for m in (policy, ref_policy, reward_model, value_model):
    m.config.pad_token_id = tokenizer.pad_token_id

print("PPO용 4개 모델 준비 완료")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/77 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: sft_distilgpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PPO용 4개 모델 준비 완료


In [20]:
# PPO 학습용 데이터
ppo_questions = [q for q, _ in QA] + [q for q, _, _ in PREF]

ppo_dataset = Dataset.from_list([
    {"input_ids": tokenizer(PROMPT_TEMPLATE.format(q=q))["input_ids"]}
    for q in ppo_questions
])
print(ppo_dataset)

Dataset({
    features: ['input_ids'],
    num_rows: 12
})


In [21]:
# PPO 설정
ppo_config = PPOConfig(
    output_dir="ppo_distilgpt2",
    per_device_train_batch_size=2,
    num_mini_batches=1,
    num_ppo_epochs=2,
    total_episodes=48,          # 48 episodes / batch 2 = 24 업데이트
    response_length=24,         # 생성 길이
    temperature=0.7,
    kl_coef=0.05,               # KL 페널티 계수 β
    missing_eos_penalty=1.0,    # EOS 없이 끝나면 감점
    stop_token="eos",
    learning_rate=1e-5,
    logging_steps=1,
    report_to="none",
    bf16=False, fp16=False,
    gradient_checkpointing=False,
)

# PPO 학습
ppo_trainer = PPOTrainer(
    args=ppo_config,
    processing_class=tokenizer,
    model=policy,
    ref_model=ref_policy,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=ppo_dataset,
)
ppo_trainer.train()

[transformers] Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate', 'output_scores'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


===training policy===


Step,Training Loss


[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, bu

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### PPO 정책 결과 확인

PPO 이후 정책이 더 높은 점수의 답변을 내는지 확인합니다.

In [22]:
policy.eval()
reward_model.eval()

print("PPO — 보상모델 점수 \n")
for q in ["Can I get a refund?", "The app keeps crashing.", "Where is my order?"]:
    before = generate(ref_policy, q)   # 고정된 SFT 정책
    after  = generate(policy, q)       # PPO로 갱신된 정책
    print(f"Q: {q}")
    print(f"  SFT ({reward_score(q, before, reward_model):+.2f}): {before}")
    print(f"  PPO ({reward_score(q, after,  reward_model):+.2f}): {after}\n")

PPO — 보상모델 점수 

Q: Can I get a refund?
  SFT (+0.66): Sure! Thank you.
  PPO (+4.78): Let us check the status of your refund.

Q: The app keeps crashing.
  SFT (+2.81): We're sorry about that.
  PPO (-0.84): No problem!

Q: Where is my order?
  SFT (-1.87): Where is my order?
  PPO (+2.98): I'm sorry about that.

